# N01 - Build and freeze all P0 + E1 data (Batch A)
Runs on the **PC** (CPU only, about 20-40 min the first time, mostly downloads).
Equivalent terminal command: `python scripts/freeze_data.py --config configs/data/freeze_all.yaml`.

Prerequisite: `python scripts/pin_revisions.py` has written `configs/revisions.lock.yaml`.
Outputs: `data/frozen/...`, `data/derived/{P0,E1}/s*/...` (git-ignored, INTERNAL) and
`results/{P0,E1}/manifests/` (tracked).

In [ ]:
import sys, os
from pathlib import Path
REPO = Path(os.environ.get("HEIRLOOM_REPO", Path.cwd().parents[1]))
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print("repo:", REPO)

In [ ]:
from heirloom.data import sources
for kind, reg in (("dataset", sources.dataset_registry()), ("model", sources.model_registry())):
    for name in reg:
        try:
            rev = sources.dataset_revision(name) if kind == "dataset" else sources.model_revision(name)
            print(f"pinned   {kind:7s} {name:26s} {rev[:12]}")
        except sources.UnpinnedRevisionError:
            print(f"UNPINNED {kind:7s} {name}  -> run scripts/pin_revisions.py")

In [ ]:
import time
from heirloom.data.build_dataset import run_freeze
t0 = time.time()
manifest = run_freeze(REPO / "configs/data/freeze_all.yaml")
print(f"{(time.time() - t0) / 60:.1f} min")

In [ ]:
import pandas as pd
rep = manifest["reports"]
print("PKU drops:", rep["prepare"]["drops"])
print("rows per split after dedup:", rep["prepare"]["rows_per_split"])
print("removed for eval leakage:", rep["leakage"]["removed_total"])
print("trigger hits:", rep["trigger_scan"]["hits"], "-> absent:", rep["trigger_scan"]["absent"])
rows = []
for exp, d in manifest["derived"].items():
    for seed, s in d["seeds"].items():
        rows.append({"exp": exp, "seed": seed, "rows": s["n_rows"], "poisoned": s["n_poisoned"],
                     "correspondence": s["correspondence"]["passed"], "index_hash": s["index_set_hash"][:12],
                     **{f"len_{k}": v for k, v in s["approx_tokens_prompt_plus_longer_response"].items()}})
pd.DataFrame(rows)

**Done when:** every seed shows `correspondence == True`, `poisoned` = 200 (P0) / 1,000 (E1),
trigger hits are all zero, and `pytest -q` passes. Then commit `configs/revisions.lock.yaml` and
`results/*/manifests/` (never `data/`).